In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import random

from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder

SEED_VALUE = 42
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)

2024-02-24 09:09:14.381087: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-02-24 09:09:14.434404: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-24 09:09:14.434443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-24 09:09:14.435524: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-24 09:09:14.448510: I tensorflow/core/platform/cpu_feature_guar

# Load Data and Preparation

In [2]:
df = pd.read_csv("./credit_risk_dataset.csv")
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [3]:
df.isna().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [4]:
df.dropna(inplace=True)

In [5]:
df.duplicated().sum()

137

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
num_cols = []
cat_cols = []

for col in df.columns :
    if df[col].dtype == 'object' :
        cat_cols.append(col)
    else : 
        num_cols.append(col)

print(num_cols)
print(cat_cols)

['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_cred_hist_length']
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


In [8]:
ord_cols = ['loan_grade']
nom_cols = ['person_home_ownership', 'loan_intent']
bin_cols = ['cb_person_default_on_file']

# Split Data

In [9]:
x, y = df.drop(columns=['loan_amnt'], axis=1), df['loan_amnt']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=SEED_VALUE)
x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, test_size=0.5, random_state=SEED_VALUE)

In [10]:
print(x_train.shape,y_train.shape)

(17100, 11) (17100,)


# Data Preproccessing

In [11]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
new_nom_cols = ohe.fit_transform(x_train[nom_cols])
encoded = pd.DataFrame(new_nom_cols, columns=ohe.get_feature_names_out(nom_cols))
x_train.reset_index(drop=True)
x_train = pd.concat([x_train.drop(nom_cols, axis=1, errors='ignore'), encoded], axis=1)

In [12]:
#x_train.dropna(inplace=True)
x_train

,person_age,person_income,person_emp_length,loan_grade,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
1856,21.0,30000.0,0.0,B,11.71,0.0,0.08,N,3.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1407,23.0,45000.0,0.0,C,14.26,0.0,0.02,Y,2.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
7250,23.0,30000.0,0.0,C,14.35,0.0,0.21,N,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2567,23.0,33000.0,3.0,D,15.99,0.0,0.25,N,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5922,24.0,24000.0,0.0,C,12.73,0.0,0.23,N,4.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17093,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
17096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
17097,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
17098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [13]:
x_train.drop(ord_cols, errors='ignore', inplace=True, axis=1)
x_train.drop(bin_cols, errors='ignore', inplace=True, axis=1)

In [14]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

2024-02-24 09:09:18.568328: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2e:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-02-24 09:09:18.633103: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2e:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-02-24 09:09:18.633245: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2e:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-02-24 09:09:18.641292: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2e:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-02-24 09:09:18.641522: I external/local_xla/xla/stream_executor

ValueError: Dimensions 25238 and 17100 are not compatible

# Create Model

In [15]:
model = tf.keras.Sequential([
    Dense(5, activation='relu', input_shape=(11,)),
    Dense(3, activation='relu'),
    Dense(1)
])

In [16]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 5)                 60        
                                                                 
 dense_1 (Dense)             (None, 3)                 18        
                                                                 
 dense_2 (Dense)             (None, 1)                 4         
                                                                 
Total params: 82 (328.00 Byte)
Trainable params: 82 (328.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
x = tf.constant([[i for i in range(11)]])

In [20]:
model.predict(x)

2024-02-24 09:10:01.926820: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


1/1 [==============================] - 1s 1s/step


array([[3.4877787]], dtype=float32)